In [ ]:
import modal

app = modal.App("rag-orchestrator")

# Definition for a lightweight CPU environment for query embedding
cpu_image = modal.Image.debian_slim().pip_install("sentence-transformers")
# Definition for a high-performance GPU environment for heavy cross-encoding
gpu_image = modal.Image.debian_slim().pip_install("torch", "transformers", "sentence-transformers")


@app.function(image=cpu_image)
def retrieve_candidates(query: str):
    # 1. Load a lightweight embedding model on CPU
    # 2. Convert query to vector
    # 3. Query your external vector database (e.g., Qdrant/Elastic)
    # 4. Return top 100 candidates
    pass


@app.function(image=gpu_image, gpu="L4")
def rerank_documents(query: str, candidates: list):
    # 1. Boot up a fast L4 GPU container
    # 2. Feed query + 100 documents into BGA-Reranker-Large
    # 3. Sort and slice down to the top 30-40 hyper-relevant documents
    pass


@app.function(image=gpu_image, gpu="A100")
def generate_response(query: str, ranked_docs: list):
    # 1. Spin up an A100 to handle the huge 20k token KV cache
    # 2. Run your generator (e.g., Llama-3-8B via vLLM)
    # 3. Stream the final answer back to your frontend
    pass


@app.function(image=cpu_image)
@modal.web_endpoint()
def orchestrate_rag_pipeline(user_query: str):
    # The clean workflow master controller
    candidates = retrieve_candidates.local(user_query)
    best_docs = rerank_documents.local(user_query, candidates)
    return generate_response.remote_gen(user_query, best_docs)

In [ ]:
# import modal
# from fastapi.responses import StreamingResponse

# app = modal.App("async-rag-pipeline")


# @app.function()
# def retrieve_docs(query: str) -> list:
#     # Standard, clean, blocking vector search logic
#     return ["doc1_chunk...", "doc2_chunk...", "etc..."]


# @app.function(gpu="L4")
# def rerank_docs(query: str, docs: list) -> list:
#     # Heavy tensor matrix calculations on GPU
#     return docs[:40]  # Returns top 40 sorted chunks


# @app.function(gpu="A100")
# async def generate_stream(query: str, context_docs: list):
#     # This must be async to yield tokens over the wire as they appear
#     # Simulated token emission from an LLM engine:
#     for token in ["This ", "is ", "the ", "grounded ", "answer..."]:
#         yield token


# @app.function()
# @modal.concurrent(max_inputs=64)  # Allows 64 users to hit this container concurrently
# @modal.web_endpoint()
# async def endpoint(user_query: str):
#     # 1. Fetch chunks (runs synchronously)
#     candidates = retrieve_docs.local(user_query)

#     # 2. Trim down to 40 (runs synchronously on GPU)
#     best_40 = rerank_docs.local(user_query, candidates)

#     # 3. Stream back to the frontend asynchronously
#     # Notice we use .remote_gen.aio() to handle the async generator stream
#     async def response_generator():
#         async for token in generate_stream.remote_gen.aio(user_query, best_40):
#             yield token

#     return StreamingResponse(response_generator(), media_type="text/plain")

In [ ]:
# import os

# import modal

# search_core_mount = modal.Mount.from_local_dir(
#     local_path="../../packages/search_core/search_core",
#     remote_path="/root/search_core"
# )

# image = (
#     modal.Image.debian_slim(python_version="3.12")
#     .pip_install("qdrant-client", "pydantic>=2.0.0", "torch", "transformers")
# )

# app = modal.App("async-rag-pipeline", image=image, mounts=[search_core_mount])

# # Core library models
# from search_core.models import SearchConfig, SearchMode, SearchQuery
# from search_core.stores import QdrantEmbeddingStore, QdrantStoreConfig


# @app.cls()  # 1. Convert to a class for container persistent state
# class DocumentRetriever:
#     @modal.enter()  # Runs ONCE when container starts up
#     def setup(self):
#         qdrant_url = os.environ.get("VECTOR_DB_URL", "http://localhost:6333")
#         config = QdrantStoreConfig(url=qdrant_url, collection_name="documents")

#         # Connection stays open and warm in memory
#         self.store = QdrantEmbeddingStore(config=config)

#     @modal.method()  # Exposes this as an invokable remote method
#     def retrieve(self, user_query: str) -> list:
#         query_obj = SearchQuery(
#             text=user_query,
#             config=SearchConfig(limit=50, mode=SearchMode.HYBRID)
#         )
#         search_response = self.store.search(query_obj)
#         return [hit.document.content for hit in search_response.results]


# @app.cls(gpu="L4")  # 2. Keep the heavy GPU model loaded in VRAM
# class DocumentReranker:
#     @modal.enter()
#     def load_model(self):
#         from search_core.rerankers import Reranker
#         # Model weights load into GPU memory ONCE
#         self.reranker = Reranker(model_name="BAAI/bge-reranker-large")

#     @modal.method()
#     def rerank(self, query: str, docs: list) -> list:
#         ranked_results = self.reranker.rerank(query=query, documents=docs)
#         return [item.document for item in ranked_results[:40]]


# @app.cls(gpu="A100")
# class LLMEngine:
#     @modal.enter()
#     def load_llm(self):
#         # TODO: Initialize your vLLM engine or weights here
#         pass

#     @modal.method()
#     async def generate_stream(self, query: str, context_docs: list):
#         simulated_tokens = ["Based ", "on ", "the ", "retrieved ", "context... "]
#         for token in simulated_tokens:
#             yield token


# # --- Orchestration Endpoint ---

# @app.function()
# @modal.concurrent(max_inputs=64)
# @modal.web_endpoint()
# async def endpoint(user_query: str):
#     # Instantiate the classes (Modal maps these to warm container instances)
#     retriever = DocumentRetriever()
#     reranker = DocumentReranker()
#     llm = LLMEngine()

#     # .remote() execution reuses the warmed up class instances
#     candidates = retriever.retrieve.remote(user_query)
#     best_40 = reranker.rerank.remote(user_query, candidates)

#     async def response_generator():
#         # Notice we use .remote_gen.aio() for async generator streams
#         async for token in llm.generate_stream.remote_gen.aio(user_query, best_40):
#             yield token

#     return StreamingResponse(response_generator(), media_type="text/plain")